<a href="https://colab.research.google.com/github/enilt/langgraph-alura/blob/main/Aula_04_Persist_Stream_Groq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 04 — Persistência e Streaming (adaptado para **Groq**)

Nesta aula o agente ganha **memória entre perguntas** usando um *checkpointer* em **SQLite**, e vemos o grafo rodando em **streaming**.

**O que mudou em relação ao original (Google Gemini):**
- LLM: `gemini-1.5-pro` → **Groq** `openai/gpt-oss-120b`.
- Chaves via `get_secret` (funciona no **Colab** e no **VS Code**), com `.strip()`.
- Prompts e perguntas **preservados** como no curso original.

## 1. Instalação dos pacotes

Trocamos `langchain-google-genai` por `langchain-groq`. O pacote `langgraph-checkpoint-sqlite` é o que fornece o `SqliteSaver` (persistência).

In [1]:
%pip install -q python-dotenv
%pip install -q -U langgraph langchain langchain-groq langchain-tavily aiosqlite langgraph-checkpoint-sqlite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.4/163.4 kB 10.4 MB/s eta 0:00:00


## 2. Chaves de API (Groq + Tavily)

O `get_secret` procura a chave em 3 lugares: **variável de ambiente → Colab Secrets (🔑) → arquivo `.env`**. O `.strip()` remove espaços/quebras de linha invisíveis que causam erro de header na Tavily.

> **Colab:** cadastre `GROQ_API_KEY` e `TAVILY_API_KEY` no 🔑 e ligue o **Notebook access**.
> **VS Code:** coloque-as num arquivo `.env` na mesma pasta.

In [2]:
import os

def get_secret(nome):
    # 1) Variavel de ambiente
    valor = os.environ.get(nome)
    if valor:
        return valor.strip()
    # 2) Colab Secrets (icone de chave na barra lateral)
    try:
        from google.colab import userdata
        valor = userdata.get(nome)
        if valor:
            return valor.strip()
    except Exception:
        pass
    # 3) Arquivo .env (VS Code / local)
    try:
        from dotenv import load_dotenv
        load_dotenv()
        valor = os.environ.get(nome)
        if valor:
            return valor.strip()
    except Exception:
        pass
    raise RuntimeError(
        f"Nao encontrei o segredo '{nome}'. Configure GROQ_API_KEY e TAVILY_API_KEY "
        f"como variavel de ambiente, Colab Secret ou no arquivo .env."
    )

# Chaves necessarias: Groq (LLM) e Tavily (busca na web)
os.environ["GROQ_API_KEY"]   = get_secret("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"] = get_secret("TAVILY_API_KEY")

MODELO_GROQ = "openai/gpt-oss-120b"
print("Chaves carregadas. Modelo:", MODELO_GROQ)

Chaves carregadas. Modelo: openai/gpt-oss-120b


## 3. Imports e persistência (checkpointer em SQLite)

O `SqliteSaver` guarda o estado do grafo num banco `checkpoints.db`. É isso que dá **memória** ao agente entre as perguntas de um mesmo `thread_id`.

In [3]:
import operator
from typing import Annotated, List, Any, Dict
from dataclasses import dataclass, field
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage, BaseMessage, AnyMessage

from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch

from langgraph.checkpoint.sqlite import SqliteSaver

from typing_extensions import TypedDict

import sqlite3

In [4]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

conn = sqlite3.connect("checkpoints.db", check_same_thread=False)
memory = SqliteSaver(conn)

## 4. Definição do agente (grafo LangGraph)

O grafo tem dois nós: `llm` (o modelo decide) e `action` (executa a ferramenta). O ciclo se repete até o modelo não pedir mais ferramentas — o padrão **ReAct**.

In [5]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system

        graph = StateGraph(AgentState)

        graph.add_node("llm", self.call_model)

        graph.add_node("action", self.take_action)

        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})

        graph.add_edge("action", "llm")

        graph.set_entry_point("llm")

        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_model(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages

        print("Mensagens enviadas ao modelo:", messages)
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling Tool: {t['name']} with args: {t['args']}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Returning to LLM after action!")
        return {'messages': results}

## 5. Ferramenta de busca + modelo Groq + prompt

Usamos a busca **Tavily** e o modelo **Groq** com `reasoning_effort="low"`. O `prompt_system` é o **mesmo do curso original**.

In [6]:
current_tavily_api_key = os.getenv('TAVILY_API_KEY')
if not current_tavily_api_key:
    raise ValueError("TAVILY_API_KEY não encontrada. Certifique-se de que está no seu .env e python-dotenv está instalado.")

tool = TavilySearch(max_results=3, tavily_api_key=current_tavily_api_key)


prompt_system = """Você é um assistente de pesquisa inteligente. Use o mecanismo de busca (tavily_search_results_json) para procurar informações.
Você tem permissão para fazer múltiplas chamadas à ferramenta (em conjunto ou em sequência).
Busque informações apenas quando tiver certeza do que procurar.
Se precisar de mais detalhes para formular uma pergunta de acompanhamento, você tem permissão para fazer isso.
Quando solicitado a comparar informações (ex: qual é mais quente, maior, etc.), use as informações do histórico da conversa e dos resultados das ferramentas."""

model = ChatGroq(model=MODELO_GROQ, temperature=0, reasoning_effort="low")

abot = Agent(model, [tool], system=prompt_system, checkpointer=memory)

## 6. Testando a persistência (mesmo `thread_id` lembra o contexto)

As 3 perguntas abaixo usam `thread_id = "1"`. Repare que a 2ª ("E no Rio de Janeiro?") e a 3ª ("Qual está mais quente?") **só fazem sentido porque o agente lembra** das respostas anteriores — graças ao checkpointer.

In [7]:
messages = [HumanMessage(content="Como está o tempo em São Paulo hoje (21/08/2025)?")]
thread = {"configurable": {"thread_id": "1"}}


print("\n--- Pergunta 1: Tempo em São Paulo ---")
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        if k in ("llm", "action"):
             print(f"{k}: {v['messages']}")


--- Pergunta 1: Tempo em São Paulo ---
Mensagens enviadas ao modelo: [SystemMessage(content='Você é um assistente de pesquisa inteligente. Use o mecanismo de busca (tavily_search_results_json) para procurar informações.\nVocê tem permissão para fazer múltiplas chamadas à ferramenta (em conjunto ou em sequência).\nBusque informações apenas quando tiver certeza do que procurar.\nSe precisar de mais detalhes para formular uma pergunta de acompanhamento, você tem permissão para fazer isso.\nQuando solicitado a comparar informações (ex: qual é mais quente, maior, etc.), use as informações do histórico da conversa e dos resultados das ferramentas.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Como está o tempo em São Paulo hoje (21/08/2025)?', additional_kwargs={}, response_metadata={})]
llm: [AIMessage(content='', additional_kwargs={'reasoning_content': 'Need current weather. Use search.', 'tool_calls': [{'id': 'fc_efc83dc1-ddeb-4514-b0d0-44b62e8312a8', 'function': {

In [8]:
messages = [HumanMessage(content="E no Rio de Janeiro?")]
thread = {"configurable": {"thread_id": "1"}}

print("\n--- Pergunta 2: Tempo no Rio de Janeiro ---")
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        if k in ("llm", "action"):
            print(f"{k}: {v['messages']}")


--- Pergunta 2: Tempo no Rio de Janeiro ---
Mensagens enviadas ao modelo: [SystemMessage(content='Você é um assistente de pesquisa inteligente. Use o mecanismo de busca (tavily_search_results_json) para procurar informações.\nVocê tem permissão para fazer múltiplas chamadas à ferramenta (em conjunto ou em sequência).\nBusque informações apenas quando tiver certeza do que procurar.\nSe precisar de mais detalhes para formular uma pergunta de acompanhamento, você tem permissão para fazer isso.\nQuando solicitado a comparar informações (ex: qual é mais quente, maior, etc.), use as informações do histórico da conversa e dos resultados das ferramentas.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Como está o tempo em São Paulo hoje (21/08/2025)?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'reasoning_content': 'Need current weather. Use search.', 'tool_calls': [{'id': 'fc_efc83dc1-ddeb-4514-b0d0-44b62e8312a8', 'function': {'

In [9]:
messages = [HumanMessage(content="Qual está mais quente?")]
thread = {"configurable": {"thread_id": "1"}}

print("\n--- Pergunta 3: Comparação ---")
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        if k in ("llm", "action"):
            print(f"{k}: {v['messages']}")


--- Pergunta 3: Comparação ---
Mensagens enviadas ao modelo: [SystemMessage(content='Você é um assistente de pesquisa inteligente. Use o mecanismo de busca (tavily_search_results_json) para procurar informações.\nVocê tem permissão para fazer múltiplas chamadas à ferramenta (em conjunto ou em sequência).\nBusque informações apenas quando tiver certeza do que procurar.\nSe precisar de mais detalhes para formular uma pergunta de acompanhamento, você tem permissão para fazer isso.\nQuando solicitado a comparar informações (ex: qual é mais quente, maior, etc.), use as informações do histórico da conversa e dos resultados das ferramentas.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Como está o tempo em São Paulo hoje (21/08/2025)?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'reasoning_content': 'Need current weather. Use search.', 'tool_calls': [{'id': 'fc_efc83dc1-ddeb-4514-b0d0-44b62e8312a8', 'function': {'arguments': '

## 7. Thread diferente = sem memória

Agora a **mesma** pergunta ("Qual está mais quente?"), mas com `thread_id = "2"`. Como é outra conversa, o agente **não tem o histórico** — é a prova de que a persistência é por thread.

In [10]:
messages = [HumanMessage(content="Qual está mais quente?")]
thread = {"configurable": {"thread_id": "2"}}

print("\n--- Pergunta 4: Comparação ---")
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        if k in ("llm", "action"):
            print(f"{k}: {v['messages']}")


--- Pergunta 4: Comparação ---
Mensagens enviadas ao modelo: [SystemMessage(content='Você é um assistente de pesquisa inteligente. Use o mecanismo de busca (tavily_search_results_json) para procurar informações.\nVocê tem permissão para fazer múltiplas chamadas à ferramenta (em conjunto ou em sequência).\nBusque informações apenas quando tiver certeza do que procurar.\nSe precisar de mais detalhes para formular uma pergunta de acompanhamento, você tem permissão para fazer isso.\nQuando solicitado a comparar informações (ex: qual é mais quente, maior, etc.), use as informações do histórico da conversa e dos resultados das ferramentas.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Qual está mais quente?', additional_kwargs={}, response_metadata={})]
llm: [AIMessage(content='Parece que a sua pergunta está incompleta — não está claro a que itens ou lugares você está se referindo quando pergunta **“Qual está mais quente?”**. Poderia, por favor, especificar quais são 